# H3 Geospatial Join

This notebook performs the same ZCTA ↔ NWS Forecast Zone join as the other three engines,
but uses **Uber's H3 hexagonal grid** instead of continuous-geometry indexes (R-tree, GiST).

## How H3 is different

| Approach | Index type | Join mechanism |
|---|---|---|
| PostGIS / DuckDB / Sedona | R-tree bounding-box pruning + exact `ST_Intersects` | Continuous geometry test |
| **H3** | Hierarchical hexagonal grid | Integer set intersection |

Instead of indexing geometries in a tree, H3 **discretizes** each polygon into a set of
hexagonal cell IDs. The join becomes: *which ZCTAs and forecast zones share at least one
hexagonal cell?* Matching on integers is extremely fast.

## Key trade-off: approximation

Because polygons are converted to discrete hexagons, results are **approximate**:
- Cells whose centers fall outside a thin polygon edge may be missed (under-count).
- The higher the resolution, the finer the approximation — and the more cells to store/join.
- This notebook uses **resolution 8** (~460 m hex edge, ~0.74 km² per cell).
  That is fine enough for PA ZCTAs while keeping cell counts manageable.

## Install
```
pip install h3
```

In [1]:
import h3
import time
import geopandas as gpd
import pandas as pd

# =========================
#  CONFIG – EDIT THESE
# =========================
WX_FILE  = "/workspace/data/pa_subsets/pa_z_18mr25.parquet"
ZIP_FILE = "/workspace/data/pa_subsets/pa_zcta_tl_2020_us_zcta520.parquet"
OUTPUT_CSV = "/workspace/data/h3_geojoin_results.csv"

# H3 resolution: 0 (coarsest) – 15 (finest)
# Resolution 8 → avg hex area ~0.74 km², edge ~460 m
H3_RESOLUTION = 8

# =========================
#  LOAD DATA
# =========================
# H3 requires coordinates in EPSG:4326 (lat/lon)
gdf_wx  = gpd.read_parquet(WX_FILE)
gdf_zip = gpd.read_parquet(ZIP_FILE)

if gdf_wx.crs is None or gdf_wx.crs.to_epsg() != 4326:
    gdf_wx = gdf_wx.set_crs(4326, allow_override=True)
if gdf_zip.crs is None or gdf_zip.crs.to_epsg() != 4326:
    gdf_zip = gdf_zip.set_crs(4326, allow_override=True)

print(f"wxareas rows:  {len(gdf_wx)}")
print(f"zipcodes rows: {len(gdf_zip)}")

wxareas rows:  78
zipcodes rows: 1833


In [2]:
# =========================
#  H3 POLYFILL HELPER
# =========================
def polyfill(geom, res):
    """Return a frozenset of H3 cell IDs covering a Shapely geometry.

    Supports both h3-py v3 (polyfill_geojson) and v4 (geo_to_cells).
    Only cells whose centers fall inside the polygon are included
    (the default 'center' containment mode).
    """
    if geom is None or geom.is_empty:
        return frozenset()
    geo = geom.__geo_interface__   # GeoJSON dict (works for Polygon & MultiPolygon)
    try:
        # h3-py >= 4.x
        return frozenset(h3.geo_to_cells(geo, res))
    except AttributeError:
        # h3-py 3.x fallback
        return frozenset(h3.polyfill_geojson(geo, res))


# Mean area of one hexagon at the chosen resolution (m²)
try:
    H3_CELL_AREA_M2 = h3.average_hexagon_area(H3_RESOLUTION, unit="m^2")   # h3 v4
except AttributeError:
    H3_CELL_AREA_M2 = h3.hex_area(H3_RESOLUTION) * 1e6                     # h3 v3 returns km²

print(f"H3 resolution: {H3_RESOLUTION}")
print(f"Mean cell area: {H3_CELL_AREA_M2:,.0f} m²  ({H3_CELL_AREA_M2/1e6:.4f} km²)")

H3 resolution: 8
Mean cell area: 737,328 m²  (0.7373 km²)


In [3]:
# =========================
#  POLYFILL + BUILD INDEX (setup — not timed)
#  Equivalent to CREATE INDEX in the other notebooks.
# =========================

# 1. Convert each polygon to a set of H3 cell IDs
wx_cells  = {i: polyfill(row.geometry, H3_RESOLUTION) for i, row in gdf_wx.iterrows()}
zip_cells = {i: polyfill(row.geometry, H3_RESOLUTION) for i, row in gdf_zip.iterrows()}

# 2. Build inverted index: cell ID → list of wx zone row indices
cell_to_wx = {}
for i, cells in wx_cells.items():
    for c in cells:
        cell_to_wx.setdefault(c, []).append(i)

# =========================
#  COLD RUN (join only)
# =========================
t0 = time.perf_counter()

pairs = set()
for zi, z_cells in zip_cells.items():
    for c in z_cells:
        for wi in cell_to_wx.get(c, []):
            pairs.add((zi, wi))

cold_time = time.perf_counter() - t0
print(f"❄️  Cold run (join only): {cold_time:.3f} seconds")
print(f"    Pairs found: {len(pairs)}")

❄️  Cold run (join only): 0.170 seconds
    Pairs found: 2264


In [4]:
# =========================
#  WARM RUN
#  Cell sets are already computed — this is the join step only.
#  Analogous to a warm cache / hot query in the other notebooks.
# =========================
t0 = time.perf_counter()

pairs_warm = set()
for zi, z_cells in zip_cells.items():
    for c in z_cells:
        for wi in cell_to_wx.get(c, []):
            pairs_warm.add((zi, wi))

warm_time = time.perf_counter() - t0
print(f"🔥  Warm run (join only): {warm_time:.3f} seconds")
print(f"    Successful joins: {len(pairs_warm)}")

🔥  Warm run (join only): 0.244 seconds
    Successful joins: 2264


In [5]:
# =========================
#  BUILD RESULT DATAFRAME
# =========================
# area_m2_approx = number of overlapping H3 cells × mean cell area
# This is an approximation; compare with the exact ST_Intersection area from other engines.

rows = []
for zi, wi in sorted(pairs_warm):
    z = gdf_zip.loc[zi]
    w = gdf_wx.loc[wi]
    overlap_cells = len(wx_cells[wi] & zip_cells[zi])
    rows.append({
        "zipcode":         z["ZCTA5CE20"],
        "GEOID20":         z["GEOID20"],
        "weather_name":    w["NAME"],
        "cwa":             w["CWA"],
        "overlap_cells":   overlap_cells,
        "area_m2_approx":  overlap_cells * H3_CELL_AREA_M2,
    })

df_result = pd.DataFrame(rows)
print(f"Result rows: {len(df_result)}")
df_result.head(10)

Result rows: 2264


,zipcode,GEOID20,weather_name,cwa,overlap_cells,area_m2_approx
0,18902,18902,Upper Bucks,PHI,45,3.317974e+07
1,18902,18902,Lower Bucks,PHI,52,3.834104e+07
2,18974,18974,Lower Bucks,PHI,70,5.161293e+07
3,19013,19013,Delaware,PHI,23,1.695853e+07
4,19023,19023,Delaware,PHI,7,5.161293e+06
5,19038,19038,Eastern Montgomery,PHI,26,1.917052e+07
6,19301,19301,Eastern Chester,PHI,13,9.585259e+06
7,19317,19317,Delaware,PHI,35,2.580647e+07
8,19317,19317,Eastern Chester,PHI,44,3.244241e+07
9,19348,19348,Eastern Chester,PHI,129,9.511526e+07


In [6]:
# =========================
#  EXPORT
# =========================
df_result.to_csv(OUTPUT_CSV, index=False)
print(f"CSV written to: {OUTPUT_CSV}")

CSV written to: /workspace/data/h3_geojoin_results.csv


In [7]:
# =========================
#  SUMMARY
# =========================
print("=" * 50)
print(f"H3 resolution:          {H3_RESOLUTION}")
print(f"Mean cell area:         {H3_CELL_AREA_M2:,.0f} m²")
total_wx_cells  = sum(len(c) for c in wx_cells.values())
total_zip_cells = sum(len(c) for c in zip_cells.values())
print(f"Total wx cells:         {total_wx_cells:,}")
print(f"Total zip cells:        {total_zip_cells:,}")
print(f"Join pairs found:       {len(df_result):,}")
print(f"Cold time (join only):  {cold_time:.3f}s")
print(f"Warm time (join only):  {warm_time:.3f}s")
print("=" * 50)
print()
print("Note: H3 results are approximate. Thin polygons or small ZCTAs")
print("may be missed if no hex center falls inside them.")
print("Increase resolution to improve accuracy at the cost of more cells.")

H3 resolution:          8
Mean cell area:         737,328 m²
Total wx cells:         157,585
Total zip cells:        157,591
Join pairs found:       2,264
Cold time (join only):  0.170s
Warm time (join only):  0.244s

Note: H3 results are approximate. Thin polygons or small ZCTAs
may be missed if no hex center falls inside them.
Increase resolution to improve accuracy at the cost of more cells.
